# Parsing Tool Call Responses

When the LLM decides to use a tool, it doesn't call it directly.
It returns a special response saying:
"I want to call THIS tool with THESE arguments"

You then:
1. Parse that response
2. Call the actual Python function yourself
3. Send the result back to the LLM
4. LLM gives the final answer

This loop is the foundation of every agent.

In [13]:
import os
import json
from groq import Groq
from dotenv import load_dotenv

load_dotenv()
client = Groq(api_key=os.getenv("GROQ_API_KEY"))

# Define the actual Python functions

In [14]:
def calculate(operation: str, a: float, b: float) -> str:
    if operation == "add":
        result = a + b
    elif operation == "subtract":
        result = a - b
    elif operation == "multiply":
        result = a * b
    elif operation == "divide":
        if b == 0:
            return "Error: cannot divide by zero"
        result = a / b
    else:
        return f"Error: unknown operation: {operation}"
    return str(result)

In [22]:
def get_weather(city: str, unit: str = "celsius") -> str:
    weather_data = {
        "karachi": {"temp": 34, "condition": "Hot and humid"},
        "london": {"temp": 18, "condition": "Cloudy"},
        "new york": {"temp": 22, "condition": "Partly cloudy"},
    }
    data = weather_data.get(city.lower(), {"temp": 25, "condition": "Unknown"})
    unit_symbol = "°C" if unit == "celsius" else "°F"
    return  f"{city}: {data['temp']}{unit_symbol}, {data['condition']}"

In [23]:
def search_web(query: str) -> str:
    return f"Search results for '{query}': [simulated results about {query}]"

# Mapping function names to actual functions


In [24]:
available_tools = {
    "calculate": calculate,
    "get_weather": get_weather,
    "search_web": search_web
}

print("Function ready:", list(available_tools.keys()))

Function ready: ['calculate', 'get_weather', 'search_web']


# Tool schemas (from previous notebook)

In [18]:
tools = [
    {
        "type": "function",
        "function": {
            "name": "calculate",
            "description": "Performs basic math operations. Use this whenever the user asks to calculate, compute, or solve a math problem.",
            "parameters": {
                "type": "object",
                "properties": {
                    "operation": {"type": "string", "enum": ["add", "subtract", "multiply", "divide"]},
                    "a": {"type": "number", "description": "First number"},
                    "b": {"type": "number", "description": "Second number"}
                },
                "required": ["operation", "a", "b"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "get_weather",
            "description": "Gets current weather for a city. Use when user asks about weather or temperature.",
            "parameters": {
                "type": "object",
                "properties": {
                    "city": {"type": "string"},
                    "unit": {"type": "string", "enum": ["celsius", "fahrenheit"]}
                },
                "required": ["city"]
            }
        }
    }
]


# The tool calling loop

In [25]:
def run_agent(user_message: str) -> str:
    messages = [
        {"role": "system", "content": "You are a helpful assistant with access to tools."},
        {"role": "user", "content": user_message}
    ]

    # Step 1: Ask LLM (it may decide to use a tool
    response = client.chat.completions.create(
        model="openai/gpt-oss-20b",
        messages= messages,
        tools=tools,
        temperature=0
    )

    message = response.choices[0].message

    if message.tool_calls:
        for tool_call in message.tool_calls:
            tool_name = tool_call.function.name
            tool_args = json.loads(tool_call.function.arguments)

            print(f"LLM wants to call: {tool_name}")
            print(f"With arguments: {tool_args}")

            tool_result = available_tools[tool_name](**tool_args)
            print(f"Tool result: {tool_result}")

            messages.append({"role": "assistant", "tool_calls": message.tool_calls})
            messages.append({
                "role": "tool",
                "tool_call_id": tool_call.id,
                "content": tool_result
            })

        final_response = client.chat.completions.create(
            model="openai/gpt-oss-20b",
            messages= messages,
            temperature=0
        )
        return final_response.choices[0].message.content

    return message.content

# Testing

In [26]:
print(run_agent("What is 847 multiplied by 23"))
print("---")
print(run_agent("What is the weather in Karachi?"))
print("---")
print(run_agent("What is the capital of France"))

LLM wants to call: calculate
With arguments: {'a': 847, 'b': 23, 'operation': 'multiply'}
Tool result: 19481
The product of 847 and 23 is **19,481**.
---
LLM wants to call: get_weather
With arguments: {'city': 'Karachi', 'unit': 'celsius'}
Tool result: Karachi: 34°C, Hot and humid
**Karachi Weather**  
- Temperature: **34 °C**  
- Condition: Hot and humid.
---
The capital of France is **Paris**.
